# SQUEEZE_BULL on ETH (2026-09)

**Question (roadmap §3 research item 1, second assets):** does the shipped SQUEEZE_BULL rule — a −2 % 4-hour
open-interest drop with price down, bought at the bar's close in a bull regime, stop −2 %, target +3 %, 48-hour
time stop — earn on ETH, with the frozen June engine run unchanged on ETH hourly bars and close-of-hour open
interest built from on-disk panels?

This notebook checks the frozen run: it reloads `results/report.json` and `results/ledger_ETH.csv`, recomputes the
bull-gated statistics from the ledger and asserts they match the report. Design: [README.md](README.md) (frozen,
`results/freeze_F0.json`). Verdict and reading: [findings.md](findings.md).

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
HERE = Path.cwd(); sys.path.insert(0, str(HERE))
import sqb_eth_lib as L
R = HERE / "results"
report = json.loads((R / "report.json").read_text())
f0 = json.loads((R / "freeze_F0.json").read_text())
print("frozen", f0["created_utc"], "| outcome", report["created_utc"])
print("P1 fidelity:", {k: v for k, v in report["P1"].items() if k not in ("only_panel", "only_ref")})
print("DECISION:", json.dumps(report["decision"], indent=1))

frozen 2026-09-19T08:27:16+00:00 | outcome 2026-09-19T08:27:17+00:00
P1 fidelity: {'common_span': ['2022-02-01 20:00:00+00:00', '2026-09-04 12:00:00+00:00'], 'panel_bull': 126, 'ref_bull': 121, 'shared_bull': 115, 'jaccard_bull': 0.8712121212121212, 'max_abs_r_diff_shared': 0.034387828651473995, 'all_regimes': {'panel': 426, 'ref': 423, 'shared': 379, 'jaccard': 0.8063829787234043}, 'pass': False}
DECISION: {
 "a_oos_n": 12,
 "a_oos_mean_R": -0.08944288443015065,
 "b_full_MAR": -0.17495794210491478,
 "c_halves": [
  -0.12044994660309286,
  -0.20601779988886573
 ],
 "d_fidelity": false,
 "verdict": "DESCRIPTIVE",
 "reason": "fidelity gate failed: the builder does not reproduce the BTC run"
}


## 1. The ledgers side by side

Bull-gated by the shipped causal regime (`ret_30d_backonly`), resolved fires, at the June 18 bp inside `r_outcome`
and re-costed at 10 bp. `BTC_reference` is the revalidation's corrected-table ledger read the same way;
`BTC_panel` is the same engine on the panel-built BTC tables (the fidelity run); `ETH` is the question.

In [2]:
rows = {}
for label, key in (("BTC reference (prod tables)", "BTC_reference"), ("BTC panel-built", "BTC_panel"), ("ETH panel-built", "ETH")):
    rec = report[key]
    for form, col in (("full, 18 bp", "full_bull"), ("full, 10 bp", "full_bull_10bp"), ("OOS 2026-04-14 →, 18 bp", "oos_bull")):
        s = rec.get(col)
        if not s or not s.get("n"):
            rows[f"{label} · {form}"] = {"n": 0 if s else "—"}; continue
        rows[f"{label} · {form}"] = {"n": s["n"], "mean R": round(s["mean_R"], 3), "win": round(s["WR"], 2),
                                     "cum R": round(s["cum_R"], 1), "max DD R": round(s["maxDD"], 2),
                                     "annual R": round(s["annual_R"], 1),
                                     "MAR": None if s["MAR"] is None else round(s["MAR"], 2),
                                     "halves": f"{s['first_half_mean_R']:+.2f} / {s['second_half_mean_R']:+.2f}" if s.get("second_half_mean_R") is not None else None,
                                     "DSR": None if s.get("dsr") is None else round(s["dsr"], 3),
                                     "exits": s["exit_mix"], "span": s["first_fire"][:10] + " → " + s["last_fire"][:10]}
pd.DataFrame(rows).T

,n,mean R,win,cum R,max DD R,annual R,MAR,halves,DSR,exits,span
"BTC reference (prod tables) · full, 18 bp",122,0.215,0.57,26.2,-5.42,5.9,1.09,+0.14 / +0.29,0.983,"{'target': 49, 'stop': 43, 'tif': 30}",2022-03-25 → 2026-09-04
"BTC reference (prod tables) · full, 10 bp",—,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"BTC reference (prod tables) · OOS 2026-04-14 →, 18 bp",10,0.234,0.7,2.3,-1.15,6.3,5.49,+0.66 / -0.19,0.764,"{'tif': 5, 'target': 3, 'stop': 2}",2026-04-21 → 2026-09-04
"BTC panel-built · full, 18 bp",126,0.191,0.56,24.0,-5.39,5.3,0.99,+0.14 / +0.24,0.973,"{'target': 49, 'stop': 45, 'tif': 32}",2022-03-01 → 2026-09-04
"BTC panel-built · full, 10 bp",126,0.231,0.56,29.1,-5.03,6.4,1.28,+0.18 / +0.28,0.99,"{'target': 49, 'stop': 45, 'tif': 32}",2022-03-01 → 2026-09-04
"BTC panel-built · OOS 2026-04-14 →, 18 bp",10,0.235,0.7,2.3,-1.15,6.3,5.5,+0.66 / -0.19,0.764,"{'tif': 5, 'target': 3, 'stop': 2}",2026-04-21 → 2026-09-04
"ETH panel-built · full, 18 bp",174,-0.163,0.39,-28.4,-35.77,-6.3,-0.17,-0.12 / -0.21,0.039,"{'stop': 105, 'target': 58, 'tif': 11}",2022-02-26 → 2026-09-11
"ETH panel-built · full, 10 bp",174,-0.123,0.39,-21.4,-29.49,-4.7,-0.16,-0.08 / -0.17,0.09,"{'stop': 105, 'target': 58, 'tif': 11}",2022-02-26 → 2026-09-11
"ETH panel-built · OOS 2026-04-14 →, 18 bp",12,-0.089,0.42,-1.1,-2.95,-2.6,-0.88,-0.67 / +0.49,0.407,"{'stop': 7, 'target': 4, 'tif': 1}",2026-04-14 → 2026-09-11


In [3]:
led = pd.read_csv(R / "ledger_ETH.csv", parse_dates=["ts"])
bull = L.bull_gated(led)
mine = L.stats(bull)
saved = report["ETH"]["full_bull"]
assert mine["n"] == saved["n"] and abs(mine["mean_R"] - saved["mean_R"]) < 1e-9 and abs(mine["maxDD"] - saved["maxDD"]) < 1e-9
print("recomputed the ETH bull-gated statistics from ledger_ETH.csv and matched the report")
print("ETH fires by regime:", report["ETH"]["by_regime"], "| frame:", report["ETH"]["frame"])
print("per year (bull, 18 bp):", json.dumps(saved["per_year"], indent=1))

recomputed the ETH bull-gated statistics from ledger_ETH.csv and matched the report
ETH fires by regime: {'flat_30d': 193, 'bull_30d': 174, 'bear_30d': 142} | frame: {'first': '2021-12-01 00:00:00+00:00', 'last': '2026-09-13 23:00:00+00:00', 'rows': 41940, 'grid_hours': 41952, 'missing_hours': 12, 'oi_finite_share': 1.0}
per year (bull, 18 bp): {
 "2022": {
  "n": 37,
  "mean_R": -0.10654271295737304,
  "sum_R": -3.9420803794228023
 },
 "2023": {
  "n": 47,
  "mean_R": -0.12270351010736752,
  "sum_R": -5.7670649750462735
 },
 "2024": {
  "n": 39,
  "mean_R": -0.2651388570915974,
  "sum_R": -10.340415426572298
 },
 "2025": {
  "n": 34,
  "mean_R": -0.09138166815119506,
  "sum_R": -3.1069767171406317
 },
 "2026": {
  "n": 17,
  "mean_R": -0.3085974380363759,
  "sum_R": -5.24615644661839
 }
}


## 2. The ETH equity path against BTC's

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 4))
btc = pd.read_csv(R / "ledger_BTC_panel.csv", parse_dates=["ts"])
for label, frame, color in (("BTC panel-built (bull-gated)", btc, "tab:blue"), ("ETH panel-built (bull-gated)", led, "tab:orange")):
    b = L.bull_gated(frame).sort_values("ts")
    ax.plot(b["ts"], b["r_outcome"].cumsum(), lw=1.2, label=f"{label}, {len(b)} fires", color=color)
ax.set_ylabel("cumulative R (18 bp inside)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("SQUEEZE_BULL: the frozen June engine on BTC and on ETH, from on-disk panels")
plt.show()

C:\Users\TJ5\AppData\Local\Temp\ipykernel_59044\1239750296.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. The fidelity run

The same builder and engine on the BTC panels against the corrected-table reference: the bull-gated fire sets and
the replay on the shared fires. The open-interest series differ by source (archive vs CoinDesk / native Binance),
which moves fires across the −2 % threshold; the hourly bars were assumed identical and are not on every hour.

In [5]:
f = report["P1"]
print({k: v for k, v in f.items() if k not in ("only_panel", "only_ref")})
print("only in the panel run:", f["only_panel"][:10]); print("only in the reference:", f["only_ref"][:10])

{'common_span': ['2022-02-01 20:00:00+00:00', '2026-09-04 12:00:00+00:00'], 'panel_bull': 126, 'ref_bull': 121, 'shared_bull': 115, 'jaccard_bull': 0.8712121212121212, 'max_abs_r_diff_shared': 0.034387828651473995, 'all_regimes': {'panel': 426, 'ref': 423, 'shared': 379, 'jaccard': 0.8063829787234043}, 'pass': False}
only in the panel run: ['2022-03-01 16:00:00+00:00', '2022-03-30 21:00:00+00:00', '2024-10-09 22:00:00+00:00', '2024-11-10 21:00:00+00:00', '2024-11-18 12:00:00+00:00', '2024-12-02 17:00:00+00:00', '2025-05-11 03:00:00+00:00', '2025-06-03 05:00:00+00:00', '2025-06-05 20:00:00+00:00', '2026-01-18 23:00:00+00:00']
only in the reference: ['2024-10-09 23:00:00+00:00', '2024-10-17 19:00:00+00:00', '2024-11-18 14:00:00+00:00', '2024-11-21 15:00:00+00:00', '2024-12-19 17:00:00+00:00', '2025-06-05 21:00:00+00:00']


In [6]:
print(json.dumps(report["decision"], indent=1))

{
 "a_oos_n": 12,
 "a_oos_mean_R": -0.08944288443015065,
 "b_full_MAR": -0.17495794210491478,
 "c_halves": [
  -0.12044994660309286,
  -0.20601779988886573
 ],
 "d_fidelity": false,
 "verdict": "DESCRIPTIVE",
 "reason": "fidelity gate failed: the builder does not reproduce the BTC run"
}
